# 📘 Week 4 · Day 29-30: 대회 워크플로우 · 재현성 · 제출 체크리스트

## 🎯 학습 목표
- Kaggle **Notebook 환경** 효과적으로 사용
- **Git + Kaggle**으로 실험 관리
- **Reproducibility (재현성)** 보장하는 코드
- 팀 협업 시 주의사항
- **최종 제출 전 체크리스트**

---

## 1. Kaggle Notebook 환경

### 구조
```
/kaggle/
├── input/          ← 대회 데이터 (읽기전용)
│   └── competition-name/
│       ├── train.csv
│       └── test.csv
├── working/        ← 실행 중 쓰기 가능 (20GB)
│   └── submission.csv
└── lib/            ← (선택) 커스텀 패키지 업로드
```

### 주요 설정
- Settings (우측 패널)
  - **Accelerator**: None / GPU T4×2 / GPU P100 / TPU
  - **Internet**: 인터넷 접속 여부 (켜야 pip install 가능)
  - **Persistence**: 데이터 유지 여부


In [ ]:
# Kaggle 환경에서 파일 경로 예시
import os

# 대회 데이터 로드
# train = pd.read_csv('/kaggle/input/titanic/train.csv')
# test  = pd.read_csv('/kaggle/input/titanic/test.csv')

# 모델 저장
# model.save('/kaggle/working/model.pkl')

# 제출 파일
# submission.to_csv('/kaggle/working/submission.csv', index=False)

print("Kaggle path convention:")
print("  Input (RO) : /kaggle/input/<competition-slug>/")
print("  Output (RW): /kaggle/working/")
print("  Total quota: ~20GB in /kaggle/working/")

---

## 2. Reproducibility — 재현성 보장

### 왜 중요한가
- 같은 코드 돌렸는데 점수가 다르다? → CV 불신 → 튜닝 불가
- 팀원과 결과 맞출 때 필수
- private LB shake-up 방지

### 시드 고정 완전판

In [ ]:
import random
import numpy as np
import os

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except ImportError:
        pass

seed_everything(42)
print("모든 seed 고정 완료")

### 2.1 버전 기록

In [ ]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn

info = {
    'python'     : sys.version.split()[0],
    'os'         : platform.platform(),
    'numpy'      : np.__version__,
    'pandas'     : pd.__version__,
    'sklearn'    : sklearn.__version__,
}

try:
    import lightgbm as lgb; info['lightgbm'] = lgb.__version__
except ImportError: pass
try:
    import xgboost as xgb; info['xgboost']   = xgb.__version__
except ImportError: pass
try:
    import torch;         info['torch']      = torch.__version__
except ImportError: pass

pd.Series(info)

---

## 3. 실험 관리 (Experiment Tracking)

### 3.1 가볍게: CSV 로깅

In [ ]:
import pandas as pd
from datetime import datetime
import os

def log_experiment(log_path, experiment_name, cv_score, params, notes=''):
    row = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'exp_name':  experiment_name,
        'cv_score':  cv_score,
        'params':    str(params),
        'notes':     notes
    }
    df_new = pd.DataFrame([row])

    if os.path.exists(log_path):
        df = pd.read_csv(log_path)
        df = pd.concat([df, df_new], ignore_index=True)
    else:
        df = df_new
    df.to_csv(log_path, index=False)
    return df

# 사용 예
df = log_experiment('/tmp/experiments.csv', 'LGB_baseline',
                    cv_score=0.8723,
                    params={'lr': 0.05, 'leaves': 31},
                    notes='초기 FE only')
df = log_experiment('/tmp/experiments.csv', 'LGB_with_target_enc',
                    cv_score=0.8795,
                    params={'lr': 0.05, 'leaves': 31, 'TE': True},
                    notes='K-Fold Target Encoding 추가')
df

### 3.2 프로: Weights & Biases (wandb)

```python
import wandb

wandb.init(project='titanic-comp', config={'lr':0.05, 'leaves':31})
for epoch in range(100):
    wandb.log({'train_loss': tr_loss, 'val_auc': auc})
wandb.finish()
```

### 3.3 폴더 구조 추천
```
competition-name/
├── data/                # 원본 데이터 (.gitignore)
├── notebooks/
│   ├── 01_eda.ipynb
│   ├── 02_fe.ipynb
│   ├── 03_baseline.ipynb
│   └── 99_submission.ipynb
├── src/                 # 재사용 코드
│   ├── features.py
│   ├── models.py
│   └── utils.py
├── submissions/         # 제출 파일 기록
├── outputs/             # 모델·OOF·로그
├── experiments.csv
├── requirements.txt
└── README.md
```

---

## 4. 팀 협업 가이드

### 4.1 역할 분담 예시
- **EDA 담당**: 데이터 이상치, 패턴, 인사이트
- **FE 담당**: 피처 엔지니어링, encoding
- **Modeling 담당**: 모델 튜닝, OOF 생성
- **Stacking 담당**: 각자의 OOF를 받아 앙상블

### 4.2 팀 작업 규칙
1. **동일 CV 시드·fold**로 모든 모델이 OOF 생성 (stacking 가능)
2. **OOF 공유 포맷**: `{model_name}_oof.npy`, `{model_name}_test.npy`
3. **실험 로그 공유**: Google Sheets 또는 wandb
4. **데이터/코드 Git 분리**: 데이터는 kaggle datasets, 코드는 GitHub

### 4.3 Kaggle API 팀 활용
```bash
# 데이터셋 팀 공유
kaggle datasets create -p ./processed_data -r zip
kaggle datasets version -p ./processed_data -r zip -m "v2"
```

---

## 5. 최종 제출 전 체크리스트

### 5.1 코드 검증
- [ ] **Seed 고정** 모든 random state 확인
- [ ] **Data leakage 없음**: test 정보가 학습에 안 들어갔는지
- [ ] **Pipeline 안에서 scaler fit**: KFold 내부 fit 확인
- [ ] **Target encoding**: out-of-fold 방식
- [ ] **Row 순서 매칭**: submission의 id/순서가 test와 동일

### 5.2 제출 파일 검증

In [ ]:
# 제출 파일 sanity check 함수
import pandas as pd
import numpy as np

def validate_submission(sub_path, sample_sub_path, target_col='Survived'):
    sub = pd.read_csv(sub_path)
    sample = pd.read_csv(sample_sub_path)

    checks = {
        '길이 일치'        : len(sub) == len(sample),
        '컬럼 이름 일치'   : list(sub.columns) == list(sample.columns),
        'ID 순서/값 일치'  : sub.iloc[:, 0].equals(sample.iloc[:, 0]),
        'target 결측 없음' : not sub[target_col].isna().any(),
        'target dtype 일치': sub[target_col].dtype == sample[target_col].dtype if target_col in sample.columns else True,
    }
    for k, v in checks.items():
        print(f"  [{'OK' if v else 'FAIL'}] {k}")

    if target_col in sub.columns:
        print(f"\nTarget 통계:")
        print(sub[target_col].describe())

# 사용 예시 (실제 경로로 수정)
# validate_submission('submission.csv', 'sample_submission.csv', 'Survived')
print("sanity check 함수 정의 완료")

### 5.3 Final Submission 선택 전략

Kaggle은 제출 중 **2개만** private LB 평가 대상으로 선택 가능.

#### 권장 전략
1. **CV 최고 점수 제출** (안정적 접근) ← **반드시 1개**
2. **Public LB 최고 점수 제출** (공격적 접근)

#### 왜 이 조합인가
- CV만 선택 → public LB에 overfit된 모델을 놓칠 수 있음
- LB만 선택 → private shake-up에 취약
- **둘 다 선택 → 위험 분산**

---

## 6. 시간 관리 (30일 계획)

### 첫 주 (Week 1): 탐색
- 대회 정독, 데이터 이해
- EDA, baseline 모델 (LightGBM 기본값)
- Public LB 순위 파악

### 둘째 주 (Week 2): 심화
- Feature Engineering 집중
- CV 구성 (stratified / time / group)
- 단일 모델 최적화

### 셋째 주 (Week 3): 다양성
- XGBoost, CatBoost, NN 추가
- Optuna 튜닝
- OOF 생성

### 넷째 주 (Week 4): 마무리
- Stacking / Blending
- Pseudo-labeling
- TTA, augmentation 추가 실험
- **최종 제출 선정**

> 🔥 마지막 3일은 **새로운 시도 금지**. CV가 이미 확립된 파이프라인을 **robust**하게 만드는 것만.

---

## 7. 자주 하는 실수 TOP 10

1. ❌ `StandardScaler.fit(X_all)` 후 split → **leakage**
2. ❌ 전체 데이터로 target encoding → **leakage**
3. ❌ CV 안정 안 시키고 LB 따라가기 → **shake-up**
4. ❌ Threshold 0.5 고정 → F1이면 튜닝해야
5. ❌ 제출 파일 컬럼 이름/순서 틀림 → 0점
6. ❌ Seed 고정 안 함 → 재현 불가
7. ❌ 한 가지 모델만 튜닝 → 앙상블 재료 없음
8. ❌ public LB에만 반응 → private에서 대폭락
9. ❌ Missing value 처리를 train/test 다르게 → 분포 shift
10. ❌ log 타깃으로 학습 후 **expm1 복원 깜빡함**

---

## 📝 Day 29-30 체크리스트
- [ ] seed_everything 습관화
- [ ] 실험 로그 CSV/wandb로 관리
- [ ] 제출 파일 sanity check 자동화
- [ ] Final 2개 제출 전략 이해
- [ ] 자주 하는 실수 TOP 10 체크

다음이 **마지막 노트북** — GitHub에 지금까지 만든 내용을 올리는 가이드입니다.